# SpatioAI — Phase 2: Climatology & Extreme Anomaly Detection
This notebook demonstrates:
1. Generating a multi-temporal weather dataset with synthetic extreme storm clusters.
2. Fitting and evaluating a time-aware climatological baseline (mean, std, p95, p99).
3. Computing absolute and standardized precipitation anomalies ($z$-scores).
4. Detecting candidate extreme weather binary masks.
5. Extracting connected component objects, calculating physical areas ($km^2$), centroids, and bounding boxes.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

# Setup project path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data.synthetic import generate_synthetic_weather_dataset
from src.climatology.baseline import ClimatologyBaseline
from src.climatology.anomaly import AnomalyCalculator
from src.events.detection import ExtremeDetector
from src.events.segmentation import EventSegmenter

In [ ]:
# 1. Generate multi-day historical weather dataset with localized storm anomalies
ds_history = generate_synthetic_weather_dataset(
    start_date="2024-01-01",
    end_date="2024-01-20",
    freq="6h",
    lat_min=10.0,
    lat_max=28.0,
    lat_step=0.25,
    lon_min=70.0,
    lon_max=95.0,
    lon_step=0.25,
    seed=42,
)

# Inject a localized extreme cyclonic storm over Bay of Bengal on Jan 15
storm_time = "2024-01-15T12:00:00"
if storm_time in ds_history.time.values.astype(str):
    lat_mask = (ds_history.latitude >= 18.0) & (ds_history.latitude <= 22.0)
    lon_mask = (ds_history.longitude >= 85.0) & (ds_history.longitude <= 90.0)
    ds_history["precipitation"].loc[dict(time=storm_time, latitude=lat_mask, longitude=lon_mask)] += 140.0

print(f"Generated dataset shape: {dict(ds_history.sizes)}")

In [ ]:
# 2. Build time-aware climatological baseline
baseline = ClimatologyBaseline(
    grouping_method="month_hour",
    variable_configs={
        "precipitation": {"statistics": ["mean", "std", "median", "p90", "p95", "p99"]},
        "temperature": {"statistics": ["mean", "std", "p95"]},
    },
)
baseline.fit(ds_history)
print("Fitted baseline variables:", list(baseline.baseline_ds.data_vars.keys()))

In [ ]:
# 3. Compute Standardized Anomalies and Candidate Extreme Mask
calc = AnomalyCalculator(baseline)
z_score = calc.calculate(ds_history, variable="precipitation", method="standardized")

detector = ExtremeDetector(baseline=baseline)
mask_da = detector.detect_by_standardized_anomaly(ds_history, variable="precipitation", z_threshold=2.5, min_absolute_val=25.0)

segmenter = EventSegmenter(connectivity=8, minimum_cells=4, minimum_area_km2=100.0)
labeled_da, df_events = segmenter.extract_events(mask_da, ds_history["precipitation"])

print(f"Total detected candidate events: {len(df_events)}")
display(df_events.head(10))

In [ ]:
# 4. Plot Rainfall Field, Climatology, Standardized Anomaly, and Candidate Extreme Events
target_time = storm_time if storm_time in ds_history.time.values.astype(str) else ds_history.time.values[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)

# Actual Precipitation
im0 = axes[0, 0].pcolormesh(ds_history.longitude, ds_history.latitude, ds_history["precipitation"].sel(time=target_time), cmap="Blues", shading="auto")
axes[0, 0].set_title(f"Precipitation (mm) at {target_time}")
fig.colorbar(im0, ax=axes[0, 0], label="mm")

# Climatological Mean
clim_mean = baseline.align_to_dataset(ds_history, stat_name="mean", variable="precipitation").sel(time=target_time)
im1 = axes[0, 1].pcolormesh(ds_history.longitude, ds_history.latitude, clim_mean, cmap="YlGnBu", shading="auto")
axes[0, 1].set_title("Climatological Mean (mm)")
fig.colorbar(im1, ax=axes[0, 1], label="mm")

# Standardized Anomaly (z-score)
im2 = axes[1, 0].pcolormesh(ds_history.longitude, ds_history.latitude, z_score.sel(time=target_time), cmap="RdBu_r", vmin=-3, vmax=5, shading="auto")
axes[1, 0].set_title("Standardized Anomaly (z-score)")
fig.colorbar(im2, ax=axes[1, 0], label="Standard Deviations (z)")

# Labeled Candidate Events with Bounding Boxes
im3 = axes[1, 1].pcolormesh(ds_history.longitude, ds_history.latitude, labeled_da.sel(time=target_time), cmap="tab10", shading="auto")
axes[1, 1].set_title("Segmented Candidate Weather Objects")
fig.colorbar(im3, ax=axes[1, 1], label="Event Label ID")

# Overlay Bounding Boxes for events at target_time
time_events = df_events[df_events["timestamp"] == str(target_time)]
for _, ev in time_events.iterrows():
    rect = plt.Rectangle(
        (ev["min_lon"], ev["min_lat"]),
        ev["max_lon"] - ev["min_lon"],
        ev["max_lat"] - ev["min_lat"],
        fill=False,
        edgecolor="red",
        linewidth=2,
    )
    axes[1, 1].add_patch(rect)
    axes[1, 1].text(ev["centroid_lon"], ev["centroid_lat"], ev["event_id"], color="red", fontsize=9, fontweight="bold")

plt.show()